### USPS Model: Using amount variables $x_{i,r}$

In [1]:
# import the csv data files using pandas
import pandas as pd

# create DataFrame for the deliveries
df_deliveries = pd.read_csv("deliveries.csv").set_index('Delivery')

# convert the DataFrame into dictionaries (for convenience--it is also possible to directly use df_deliveries)
Origin = {}
Destination = {}
Available = {}
LatestArrival = {}
Volume = {}
for i in df_deliveries.index:
    Origin[i] = df_deliveries['Origin'][i]  # this assigns the i-th element
    Destination[i] = df_deliveries['Destination'][i]
    Available[i] = df_deliveries['Available'][i]
    LatestArrival[i] = df_deliveries['LatestArrival'][i]
    Volume[i] = df_deliveries['Volume'][i]

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# create DataFrame for the trips
df_trips = pd.read_csv("trips.csv").set_index('Trip')

# Convert the DataFrame into dictionaries (for convenience--it is also possible to directly use df_trips).
# The following parameters are easy to define:
Cost = {}
Capacity = {}
Leave = {}
Arrive = {}
for i in df_trips.index:
    Cost[i] = df_trips['Cost'][i]
    Capacity[i] = df_trips['Capacity'][i]
    Leave[i] = df_trips['Leave'][i]
    Arrive[i] = df_trips['Arrive'][i]

In [3]:
# visits are a bit more challenging: I want this to be a set (or a list) for each trip
Visits = {}
for i in df_trips.index:
    tmpList = []
    # each trip has at least two visits
    tmpList.append(df_trips['Visit 1'][i])
    tmpList.append(df_trips['Visit 2'][i])
    # and optionally a third visit (check if not not null)
    if pd.notnull(df_trips['Visit 3'][i]):
        # note: because of the missing values (NaN), this column is interpreted as type "float".
        # to make it an integer element, I cast it as "int"        
        tmpList.append(int(df_trips['Visit 3'][i]))
    Visits[i] = tmpList

In [4]:
# for convenience, define two more index sets
Deliveries = df_deliveries.index
Trips = df_trips.index

In [5]:
# Define delivery/trip compatibility
#
# We are making some strong assumptions here: 
# - all deliveries are from 1 to 4, from 1 to 5, from 2 to 4, or from 2 to 5;
# - trips are either direct or via hub denoted by 3 (all starting from {1,2} and ending in {4,5})
# Therefore, we can simply test whether the origin and destination are part of the visits of each trip, and
# whether the Available/Leave times and LatestArrival/Arrive times are compatible.
#
# In general, we would need to have access to the direction of the trip, as well as the intermediate
# times of the visits to compile this informatio (as is done in the USPS paper).
CanAssign = {}
for i in Deliveries:
    for r in Trips:
        # (note: we can use a backslash \ to continue a statement on the next line)
        if Origin[i] in Visits[r] and Destination[i] in Visits[r] \
        and Available[i] <= Leave[r] and Arrive[r] <= LatestArrival[i]:
            CanAssign[i,r] = 1

In [14]:
CanAssign

{('d1', 1): 1,
 ('d1', 13): 1,
 ('d1', 25): 1,
 ('d2', 2): 1,
 ('d2', 3): 1,
 ('d2', 13): 1,
 ('d2', 14): 1,
 ('d2', 15): 1,
 ('d2', 26): 1,
 ('d2', 27): 1,
 ('d3', 15): 1,
 ('d4', 4): 1,
 ('d4', 28): 1,
 ('d4', 29): 1,
 ('d5', 5): 1,
 ('d5', 17): 1,
 ('d6', 6): 1,
 ('d6', 18): 1,
 ('d6', 30): 1,
 ('d7', 9): 1,
 ('d7', 21): 1,
 ('d7', 33): 1,
 ('d8', 10): 1,
 ('d8', 34): 1,
 ('d9', 10): 1,
 ('d9', 22): 1,
 ('d9', 34): 1,
 ('d10', 7): 1,
 ('d10', 19): 1,
 ('d10', 31): 1,
 ('d10', 32): 1,
 ('d10', 35): 1,
 ('d11', 8): 1,
 ('d11', 11): 1,
 ('d11', 20): 1,
 ('d11', 23): 1,
 ('d11', 32): 1,
 ('d12', 12): 1,
 ('d12', 24): 1,
 ('d12', 36): 1}

In [6]:
from docplex.mp.model import Model
mdl = Model()

In [7]:
# variables
SelectTrip = mdl.binary_var_dict(Trips, name='select trip')

# save on memory by only defining variables that can be assigned:
# use the keys from 'CanAssign' here:
Amount = mdl.continuous_var_dict(CanAssign.keys(), lb=0, name='allocate amount')

In [8]:
# objective
mdl.minimize(mdl.sum(Cost[r]*SelectTrip[r] for r in Trips))

In [9]:
# Constraints: allocate each delivery
for i in Deliveries:
    mdl.add_constraint(mdl.sum(Amount[i,r] for r in Trips if (i,r) in CanAssign.keys()) == Volume[i])

In [10]:
# Constraints: meet route capacity / linkage
for r in Trips:
    mdl.add_constraint(mdl.sum(Amount[i,r] for i in Deliveries if (i,r) in CanAssign.keys()) 
                       <= Capacity[r]*SelectTrip[r])

In [11]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.025202,status='integer optimal solution')

In [12]:
mdl.print_solution()

objective: 11150
status: OPTIMAL_SOLUTION(2)
  "select trip_5"=1
  "select trip_7"=1
  "select trip_8"=1
  "select trip_10"=1
  "select trip_13"=1
  "select trip_15"=1
  "select trip_18"=1
  "select trip_21"=1
  "select trip_24"=1
  "select trip_25"=1
  "select trip_28"=1
  "select trip_34"=1
  "allocate amount_d1_13"=700.000
  "allocate amount_d1_25"=2500.000
  "allocate amount_d2_13"=1100.000
  "allocate amount_d2_15"=1700.000
  "allocate amount_d3_15"=1300.000
  "allocate amount_d4_28"=2500.000
  "allocate amount_d5_5"=2300.000
  "allocate amount_d6_18"=1700.000
  "allocate amount_d7_21"=1650.000
  "allocate amount_d8_34"=1800.000
  "allocate amount_d9_10"=1300.000
  "allocate amount_d10_7"=2000.000
  "allocate amount_d11_8"=2100.000
  "allocate amount_d12_24"=1340.000


In [13]:
for r in Trips:
    if SelectTrip[r].solution_value >= 0.99:
        print("Trip " + str(r) + " has allocated volume " +
              str(mdl.sum(Amount[i,r] for i in Deliveries if (i,r) in CanAssign).solution_value) +
             " and capacity " + str(Capacity[r]))

Trip 5 has allocated volume 2300.0 and capacity 3000
Trip 7 has allocated volume 2000.0 and capacity 2000
Trip 8 has allocated volume 2100.0 and capacity 3000
Trip 10 has allocated volume 1300.0 and capacity 2500
Trip 13 has allocated volume 1800.0 and capacity 2500
Trip 15 has allocated volume 3000.0 and capacity 3000
Trip 18 has allocated volume 1700.0 and capacity 3000
Trip 21 has allocated volume 1650.0 and capacity 3000
Trip 24 has allocated volume 1340.0 and capacity 3000
Trip 25 has allocated volume 2500.0 and capacity 2500
Trip 28 has allocated volume 2500.0 and capacity 2500
Trip 34 has allocated volume 1800.0 and capacity 2500
